# RSI 2 Strategy, Backtested on Crypto
Larry Connors' 2-period RSI (relative strength index): buy a sharp two-day drop inside an uptrend, sell the bounce — and the
mirror image for shorts. Tested on every coin that had a Binance perpetual in January 2020, with
fees and funding, from 2021 to 2026. Education, not financial advice.

The rules, as Connors gives them: long when the close is above the 200-day SMA (simple moving average)
 and RSI(2) falls below 5; exit on a close above the 5-day SMA. Short when the close is
below the 200-day SMA and RSI(2) rises above 95; exit on a close below the 5-day SMA. No stop loss.

## 1. Install
freqtrade is the backtester; we call only a few of its methods.

In [ ]:
!pip install -q freqtrade==2026.10

## 2. Helpers from the earlier episodes
`binance_vision` — raw Binance data with SHA-256 hash checks (episode q00); `quant_report` — the verdict and hourly marked-to-market equity (episode q01).

In [ ]:
!curl -fsSLO https://raw.githubusercontent.com/nikshev/unstuck/main/episodes/quant-lib/binance_vision.py -O https://raw.githubusercontent.com/nikshev/unstuck/main/episodes/quant-lib/quant_report.py
from binance_vision import save_for_freqtrade, DATA
from quant_report import verdict, equity, drawdown
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 3. The coins
The same eleven as episode q02: every perpetual Binance listed by January 2020. Daily candles from 2020, so the 200-day average is ready by January 2021. Uses `save_for_freqtrade`, `DATA` (section 2).

In [ ]:
coins = ["BTC", "ETH", "BCH", "XRP", "EOS", "LTC",
         "TRX", "ETC", "LINK", "XLM", "ADA"]
for c in coins:
    save_for_freqtrade(c + "USDT", "1d", "2020-01", "2026-10")
daily = str(DATA / "{}_USDT_USDT-1d-futures.feather")
close = pd.DataFrame({c: pd.read_feather(daily.format(c))
                         .set_index("date")["close"] for c in coins})
print(close.notna().sum().to_string())                 # days of data per coin

## 4. RSI over two days
Wilder's RSI (relative strength index): average gain $G$ and average loss $L$ over $n$ days, smoothed with weight $1/n$; $RSI = 100 - \dfrac{100}{1 + G/L}$. With $n = 2$ one bad day pushes it near 0. Uses `close` (section 3).

In [ ]:
def rsi(prices, n=2):
    change = prices.diff()
    gain = change.clip(lower=0).ewm(alpha=1 / n).mean()     # average gain
    loss = (-change.clip(upper=0)).ewm(alpha=1 / n).mean()  # average loss
    return 100 - 100 / (1 + gain / loss)

btc = pd.DataFrame({"close": close["BTC"], "RSI(2)": rsi(close["BTC"])})
print(btc.loc["2026-10-01":"2026-10-12"].round(1).to_string())

## 5. How often does it fire?
Signal days per coin: long = above the 200-day SMA (simple moving average) and RSI (relative strength index) over 2 days < 5; short = below it and RSI(2) > 95. From 2021. Uses `close` (3), `rsi` (4).

In [ ]:
sma200 = close.rolling(200).mean()
r2 = close.apply(rsi)
period = slice("2021-01-02", "2026-10-31")
longs = ((close > sma200) & (r2 < 5)).loc[period].sum()
shorts = ((close < sma200) & (r2 > 95)).loc[period].sum()
print(pd.DataFrame({"long days": longs, "short days": shorts}).to_string())

## 6. Backtest settings
Isolated margin, leverage 1, ten slots of equal size — one per coin freqtrade will trade. 10,000 USDT (Tether, a dollar stablecoin). Uses `coins` (section 3).

In [ ]:
config = {
    "trading_mode": "futures",
    "margin_mode": "isolated",               # each position has its own margin
    "liquidation_buffer": 0.0,
    "stake_currency": "USDT",
    "stake_amount": "unlimited",             # account / free slots
    "dry_run_wallet": 10000, "dry_run": True,
    "max_open_trades": 10,                   # one slot per coin traded
    "timeframe": "1d", "fee": 0.0005,
    "entry_pricing": {"price_side": "same", "use_order_book": True,
                      "order_book_top": 1},
    "exit_pricing": {"price_side": "same", "use_order_book": True,
                     "order_book_top": 1},
    "exchange": {"name": "binance",
                 "pair_whitelist": [c + "/USDT:USDT" for c in coins]},
    "pairlists": [{"method": "StaticPairList"}],
}
json.dump(config, open("config.json", "w"), indent=1)

## 7. Strategy file, part 1: the settings
Connors' thresholds 5 and 95, trend filter 200 days, exit average 5 days. He uses no stop, so the stop is freqtrade's minimum, −99%.

In [ ]:
%%writefile user_data/strategies/RSI2.py
import pandas as pd
from freqtrade.strategy import IStrategy

class RSI2(IStrategy):
    LOW, HIGH, TREND, EXIT, SHORTS = 5, 95, 200, 5, True   # Connors' rules
    timeframe = "1d"
    can_short = True
    minimal_roi = {"0": 100}            # no profit target: only the exit rule
    stoploss = -0.99                    # Connors uses no stop
    startup_candle_count = 200          # the 200-day average needs 200 days

## 8. Part 2: indicators
The same RSI (relative strength index) as section 4, and the two simple moving averages. `TREND`, `EXIT` from section 7.

In [ ]:
%%writefile -a user_data/strategies/RSI2.py

    def populate_indicators(self, df, metadata):
        change = df["close"].diff()
        gain = change.clip(lower=0).ewm(alpha=1 / 2).mean()
        loss = (-change.clip(upper=0)).ewm(alpha=1 / 2).mean()
        df["rsi"] = 100 - 100 / (1 + gain / loss)              # section 4
        df["trend"] = df["close"].rolling(self.TREND).mean()   # 200 days
        df["exit_avg"] = df["close"].rolling(self.EXIT).mean() # 5 days
        return df

## 9. Part 3: entries and exits
Signals on the daily close; the order fills at the next open, 00:00 UTC — on a market that never closes, that is the same moment. Columns from section 8, settings from 7.

In [ ]:
%%writefile -a user_data/strategies/RSI2.py

    def populate_entry_trend(self, df, metadata):
        up = df["close"] > df["trend"]                 # long-term uptrend
        df.loc[up & (df["rsi"] < self.LOW), "enter_long"] = 1
        if self.SHORTS:
            df.loc[~up & (df["rsi"] > self.HIGH), "enter_short"] = 1
        return df

    def populate_exit_trend(self, df, metadata):
        # the bounce: a close back above (below) the 5-day average
        df.loc[df["close"] > df["exit_avg"], "exit_long"] = 1
        df.loc[df["close"] < df["exit_avg"], "exit_short"] = 1
        return df

## 10. Run it
2 January 2021 to 31 October 2026. Uses `config.json` (6), `RSI2` (7–9), `verdict`, `equity`, `drawdown` (2).

In [ ]:
!freqtrade backtesting --cache none -c config.json -s RSI2 --timerange 20210102-20261031 --export trades > run.log 2>&1
both, trades = verdict("RSI2")
both["drawdown, marked to market"] = drawdown(equity(trades,
                                                     start="2021-01-02"))

## 11. What a trade looks like
Win rate, average win and loss, days held — by side; then the count and the money of the wins and the losses, profit by coin and side, the worst trade and the last Bitcoin trade. Uses `trades` (section 10).

In [ ]:
trades["side"] = np.where(trades["is_short"], "short", "long")
trades["days"] = trades["trade_duration"] / 1440          # minutes -> days
trades["win"] = trades["profit_abs"] > 0
stats = trades.groupby("side").agg(
    trades=("win", "size"), win_rate=("win", "mean"),
    avg_win=("profit_abs", lambda p: p[p > 0].mean()),
    avg_loss=("profit_abs", lambda p: p[p <= 0].mean()),
    worst=("profit_abs", "min"), days=("days", "mean"))
print(stats.round(2).to_string())
won_lost = trades.groupby(["side", "win"])["profit_abs"].agg(["size", "sum"])
print(won_lost.round(0).to_string())          # how many, and how much money
by_coin = trades.pivot_table("profit_abs", "pair", "side", aggfunc="sum")
by_coin.index = by_coin.index.str.split("/").str[0]     # BTC/USDT:USDT -> BTC
print(by_coin.round(0).to_string())
def show(label, t):                                  # one trade, one line
    print(label, t["pair"].split("/")[0],
          t["open_date"].date(), t["open_rate"], "->",
          t["close_date"].date(), t["close_rate"], f"{t['profit_abs']:+.2f}")
show("worst trade:", trades.loc[trades["profit_abs"].idxmin()])
btc = trades[trades["pair"].str.startswith("BTC")]
show("last Bitcoin trade:", btc.iloc[-1])

## 12. Long only, and the looser thresholds
Two variants from the same rules: no shorts; and RSI (relative strength index) below 10 / above 90 — the levels most retellings quote. Subclasses of `RSI2` (section 7).

In [ ]:
%%writefile user_data/strategies/RSI2Variants.py
from RSI2 import RSI2

class RSI2Long(RSI2):
    SHORTS = False

class RSI2Loose(RSI2):
    LOW, HIGH = 10, 90

### Run both
Uses `config.json` (6), the two subclasses (above), `verdict`, `equity`, `drawdown` (2).

In [ ]:
runs = {}
for name in ["RSI2Long", "RSI2Loose"]:
    !freqtrade backtesting --cache none -c config.json -s {name} --timerange 20210102-20261031 --export trades > run.log 2>&1
    runs[name], t = verdict(name)
    runs[name]["drawdown, marked to market"] = drawdown(
        equity(t, start="2021-01-02"))
    runs[name + " trades"] = t

## 13. Against holding the same coins
Buy-and-hold = the ten coins freqtrade traded, equal amounts, rebalanced daily. Uses `close` (3), `trades` (10), `runs` (12), `equity`, `drawdown` (2).

In [ ]:
traded = close.drop(columns="EOS").loc["2021-01-02":]   # what freqtrade traded
hold = 10_000 * (1 + traded.pct_change().mean(axis=1).fillna(0)).cumprod()
curves = {"buy and hold, 10 coins": hold,
          "RSI 2, long and short": equity(trades, start="2021-01-02"),
          "RSI 2, long only": equity(runs["RSI2Long trades"],
                                     start="2021-01-02")}
for name, curve in curves.items():
    print(f"{name:>23}: final {curve.iloc[-1]:>8.0f}"
          f"  max drawdown {drawdown(curve):.0%}")
plt.figure(figsize=(11, 5))
for name, curve in curves.items():
    plt.plot(curve.index, curve, label=name)
plt.yscale("log"); plt.legend(); plt.title("10,000 USDT, 2021 to 2026")
plt.show()

## 14. The Sharpe ratio: return per unit of wobble
Sharpe ratio $S = \dfrac{\text{mean of the daily returns}}{\text{their standard deviation}} \times \sqrt{365}$ — the hourly account (section 13) taken once a day; risk-free rate 0: idle USDT (Tether, a dollar stablecoin) earns nothing here. Uses `curves` (13), `both` (10), `runs` (12), `equity` (2).

In [ ]:
def sharpe(curve):
    day = curve.resample("1D").last()           # the account once a day
    r = day.pct_change().dropna()               # daily returns
    return r.mean() / r.std() * 365 ** 0.5      # a year = 365 crypto days

curves["RSI 2, 10 / 90"] = equity(runs["RSI2Loose trades"],
                                  start="2021-01-02")
for name, curve in curves.items():
    r = curve.resample("1D").last().pct_change().dropna()
    print(f"{name:>23}: mean {r.mean():.3%}  std {r.std():.2%}"
          f"  Sharpe {sharpe(curve):.2f}")
both["Sharpe"] = sharpe(curves["RSI 2, long and short"])
runs["RSI2Long"]["Sharpe"] = sharpe(curves["RSI 2, long only"])
runs["RSI2Loose"]["Sharpe"] = sharpe(curves["RSI 2, 10 / 90"])

## 15. Every run side by side
A legend explains every row. Uses `both` (10, 14), `runs` (12, 14).

In [ ]:
table = pd.DataFrame({"5 / 95": both, "long only": runs["RSI2Long"],
                      "10 / 90": runs["RSI2Loose"]})
table.index = ["start", "final", "CAGR", "drawdown (freqtrade)", "trades",
               "long profit", "short profit", "funding", "fees",
               "drawdown (hourly)", "Sharpe"]
pct = ["CAGR", "drawdown (freqtrade)", "drawdown (hourly)"]
def cell(row, v):                       # how each row is written
    if row in pct:
        return f"{v:.0%}"
    return f"{v:.2f}" if row == "Sharpe" else f"{v:,.0f}"
shown_table = table.apply(lambda r: r.map(lambda v: cell(r.name, v)), axis=1)
print(shown_table.to_string())
print("money in USDT (Tether, a dollar stablecoin)")
print("5 / 95: RSI(2) thresholds; long only: no shorts")
print("CAGR = compound annual growth rate: growth per year")
print("drawdown = deepest fall from a peak; freqtrade: closed trades only,")
print("  hourly: the whole account valued every hour")
print("funding = paid between longs and shorts every 8 hours")
print("fees = the exchange fee, 0.05% of every order")
print("Sharpe = mean daily return / its standard deviation x sqrt(365);")
print("  above 1 is the usual target")